## UESL 3.2 Data Abstractions

Mission 2 of the UESL Game Maker project. In 3.1 every level had its own variable; here one list stands for all of them.


## Popcorn

**Popcorn checkpoint — extend the level sequence.**

**Prediction before running.** Starting from `[6, 9, 12]`, updating index 1 to `9 + 3` changes that one element without changing how many there are, then `append(5)` adds a fourth. So I predicted the collection `[6, 12, 12, 5]` and the count `4`.

**AP index for the same level.** Python index `1` and AP index `2` select the same element — the second level. Python counts `0, 1, 2`; College Board pseudocode counts `1, 2, 3`. For a list of length *n*, valid Python indices run `0` to `n-1` and AP indices run `1` to `n`.


In [ ]:
# CODE_RUNNER: UESL 3.2 Popcorn - Extend the level sequence

level_stars = [6, 9, 12]
level_stars[1] = level_stars[1] + 3   # replace one element, length unchanged
level_stars.append(5)                 # add one element, length grows
print(level_stars)
print(len(level_stars))


**Actual output:** `[6, 12, 12, 5]` then `4`. Matches the prediction.

**The changed version.** Appending `8` instead of `5` gives `[6, 12, 12, 8]` — still four levels. The appended *value* is not the appended *count*; one `append` adds exactly one element no matter what number goes in it.

**Why updating keeps the length and appending does not.** `level_stars[1] = ...` writes into a slot that already exists, so the number of slots is untouched. `append` creates a new slot at the end, so the length goes up by one. `insert` also adds a slot but shifts everything after it down a position, and `pop` removes a slot and closes the gap, dropping the length by one. Only the operations that add or remove slots can change `len()`.


## MCQ

**MCQ 3.2: 4/4 | answers: B,A,C,B**

| # | Question | Answer | Why |
|---|---|---|---|
| 1 | Which AP index selects the first level? | **B — 1** | AP indices run 1 through n. `0` is the Python answer, and the list length would select past the end in Python or the last element in AP. |
| 2 | Replacing one count changes the list length by | **A — 0** | Assigning into an existing slot overwrites a value. No slot is created or destroyed, so `len()` is identical before and after. |
| 3 | What does `APPEND(level_stars, 5)` do? | **C — adds one element with value 5** | The second argument is the value being added, not a position and not a count. It is the same as Python's `level_stars.append(5)`. |
| 4 | Why use a list plus traversal for 100 levels? | **B — it avoids a separate variable and sum edit for every level** | The list is not automatically faster, and it certainly does not prevent errors — an out-of-range index still crashes. What it removes is the editing work: 100 separate variables means 100 declarations and a 100-term sum to rewrite every time a level is added. |


## Homework

**Refactored from 3.1.** In 3.1 I had `level_one_stars`, `level_two_stars` and `level_three_stars`. Here that becomes `level_stars = [12, 18, 10]`, where **each element is one level's collected star count, stored in level order** — element 0 is the first level, element 1 the second, and so on. The position in the list carries the level number, so no separate variable has to.

**The edits:** add 5 to the second level, then append 7 for a brand-new level, then total the whole thing with the traversal from Example C.

**Why there is a guard on the update.** `level_stars[1]` crashes with `IndexError` on a list shorter than two elements, so the second-level update is wrapped in `if len(level_stars) >= 2:`. The traversal below it needs no guard at all — a `for` loop over an empty list simply runs zero times.

**Prediction:** `[12, 23, 10, 7]`, then `4`, then `52`.


In [ ]:
# CODE_RUNNER: UESL 3.2 Homework - Grow the level collection

# Each element is one level's collected stars, in level order.
level_stars = [12, 18, 10]

# Add 5 to the second level. Guarded: level_stars[1] would crash on a shorter list.
if len(level_stars) >= 2:
    level_stars[1] = level_stars[1] + 5

# A new level joins the game.
level_stars.append(7)

# Example C's traversal. Reset the total before calculating.
total_stars = 0
for stars in level_stars:
    total_stars = total_stars + stars

print(level_stars)
print(len(level_stars))
print(total_stars)


**Actual output:** `[12, 23, 10, 7]`, `4`, `52`. Matches the prediction.

**Why the traversal needed no new line for the fourth level.** The loop says "for every element in this list," not "for element 0, element 1, element 2." Growing the list grows the number of passes automatically. With 100 separate variables I would have to declare `level_one_stars` through `level_one_hundred_stars` and then write a 100-term addition — and every new level means editing that sum by hand, in a place far away from where the level was added. That is the maintenance work the abstraction removes, and it is the actual answer to "why a list."


## Tests

**Test 1 — the main path.** Covered by the homework cell above: `[12, 23, 10, 7]`, `4`, `52`.

**Test 2 — the empty project.** A maker who has not built a level yet. Expected `0` levels and `0` stars, with no crash. Each test starts from a fresh list rather than reusing the one above, so nothing carries over.


In [ ]:
# CODE_RUNNER: UESL 3.2 Homework - Test 2, the empty project

level_stars = []

# The guard is what keeps this from crashing - there is no second element to update.
if len(level_stars) >= 2:
    level_stars[1] = level_stars[1] + 5

total_stars = 0
for stars in level_stars:
    total_stars = total_stars + stars

print(len(level_stars))
print(total_stars)


**Actual output:** `0` then `0`. No crash.

**What the empty test proves.** `total_stars = 0` is doing real work here. The loop body never runs, so the only reason anything sensible prints is that the total was initialized before the loop. If it had been created inside the loop, this case would raise `NameError` instead — the empty list is what exposes that, which is why it is worth testing rather than assuming.


## Design Thinking

**Maker need.** A game maker wants to add a level, reorder two levels, or cut one, without hunting through the code for every place a level number was hardcoded. In 3.1 adding a fourth level meant a new variable *and* a rewritten total — two edits in two places, and forgetting the second one leaves a total that is quietly wrong rather than broken.

**My goal.** One collection in level order that can be read, updated and grown, with a total that stays correct without being rewritten.

**A representation I considered.** I thought about a dictionary keyed by level name, like `"Forest"` to `10`, which would let players and makers refer to levels by name instead of counting from zero. I stayed with a list because this lesson's operations are all positional — *the second level*, *the end of the sequence* — and level **order** is part of the game's meaning. A dictionary has no inherent order to rely on, so "the next level" would have to be reconstructed from somewhere else. The name problem is real, though, and I would solve it with a parallel list of labels rather than by giving up ordering.

**What I prototyped.** The collection above, with the guarded second-level update and the traversal total, run on both a three-level game and an empty one.

**A test and what it changed.** The empty-list test is what changed the code. My first version updated `level_stars[1]` unconditionally, which is fine on `[12, 18, 10]` and raises `IndexError` on `[]` — a brand-new project with no levels yet, which is exactly the state every maker starts in. I added the `len(level_stars) >= 2` guard because of that run. It was not a case I predicted; the test found it.

**Clear level labels.** Players should never have to know that the Cavern is index 1. The index is a storage detail, and showing it leaks the implementation into the interface — worse, it is off by one from how anyone counts out loud. Names like Forest, Cavern and Summit in the UI, with the list handling order underneath, is the same separation this whole topic is about: a name and a set of operations standing in for how the data is actually kept.

**Connection to 3.1.** Changing the list still does not recalculate a stored total. The list organizes the state; the assignment and the traversal are what update it. A total is a stored result, not a live formula — which was the whole point of 3.1 and has not stopped being true.

## Submission notes

    Lesson: UESL 3.2 Data Abstractions
    MCQ 3.2: 4/4 | answers: B,A,C,B
    Popcorn: [6, 12, 12, 5], length 4, AP index 2 selects the same level as Python index 1
    Homework: [12, 23, 10, 7], 4 levels, 52 stars
    Test 1 (main): 4 levels, 52 stars
    Test 2 (empty): 0 levels, 0 stars, no crash
